In [ ]:
import pandas as pd
from pathlib import Path

df_w = pd.read_csv("../../1-transaction-data/price_series.csv", index_col="date", parse_dates=True).asfreq("W-SUN")

folder = Path("raw_weekly_indexes")

canonical = {
    "business_bay": "Business Bay",
    "damac_hills_2": "DAMAC HILLS 2",
    "damac_hills": "DAMAC HILLS",
    "dow_square": "TOWN SQUARE",
    "downtown_dubai": "DownTown Dubai",
    "dubai_creek_harbour": "Dubai Creek Harbour",
    "dubai_marina": "Dubai Marina",
    "dubai_south_residential_district": "Dubai South Residential District",
    "dubai_sports_city": "Dubai Sports City",
    "geometry": "Al Furjan",
    "international_city_phase_1": "International City Phase 1",
    "jumeirah_lake_towers": "Jumeirah Lakes Towers",
    "jumeirah_village_circle": "Jumeirah Village Circle",
    "jumeirah_village_triangle": "Jumeirah Village Triangle",
    "jumeriah_beach_residence": "Jumeriah Beach Residence  - JBR",
    "mudon": "Mudon",
    "palm_jumeirah": "Palm Jumeirah",
    "silicon_oasis": "Silicon Oasis",
    "the_greens": "The Greens"
}

required_cols = {"week_start", "ndbi"}
base_features = ["ndbi"]
lags = [4, 8, 12, 26]

def load_region_ndbi(stem, region_name):
    csv_path = folder / f"{stem}_weekly_ndbi.csv"
    if not csv_path.exists():
        return {}
    df_sat = pd.read_csv(csv_path, parse_dates=["week_start"])
    if not required_cols.issubset(df_sat.columns):
        return {}
    out = {}
    series = (
        df_sat
        .set_index("week_start")["ndbi"]
        .astype(float)
        .sort_index()
        .resample("W-SUN").mean()
        .rolling(4, min_periods=1).mean()
        .reindex(df_w.index)
    )
    series = series.ffill().bfill()
    out[f"{region_name}__ndbi"] = series
    return out

columns_dict = {}
skipped = []

for stem, region in canonical.items():
    feats = load_region_ndbi(stem, region)
    if not feats:
        skipped.append(region)
        continue
    columns_dict.update(feats)

if not columns_dict:
    raise RuntimeError("No region NDBI series loaded; check raw_weekly_indexes/*_weekly_ndbi.csv files and headers.")

df_base = pd.DataFrame(columns_dict).reindex(df_w.index)
df_base = df_base.ffill().bfill()

feat_cols = [c for c in df_base.columns if c.endswith("__ndbi")]
if feat_cols:
    df_base["Global__ndbi"] = df_base[feat_cols].mean(axis=1)

lag_frames = []
for L in lags:
    shifted = df_base.shift(L)
    shifted.columns = [f"{c}_lag{L}" for c in shifted.columns]
    lag_frames.append(shifted)

df_all = pd.concat([df_base] + lag_frames, axis=1).reindex(df_w.index)
df_all = df_all.ffill().bfill()
df_all = df_all.apply(pd.to_numeric, errors="coerce").ffill().bfill()

out_name = "../weekly_ndbi_aligned.csv"
df_all.to_csv(out_name)
print(f"Saved {out_name} (NDBI features base + lags, fully filled).")

base_cols = [c for c in df_all.columns if "__" in c and not any(c.endswith(f"_lag{L}") for L in lags)]
lag_cols = [c for c in df_all.columns if any(c.endswith(f"_lag{L}") for L in lags)]
regions = sorted({c.split("__", 1)[0] for c in base_cols})
features_found = sorted({c.split("__", 1)[1] for c in base_cols})
nans_total = int(df_all.isna().sum().sum())
date_min = df_all.index.min()
date_max = df_all.index.max()

print("Data validation summary:")
print(f"- Date range: {date_min.date()} to {date_max.date()} | rows: {len(df_all):,} | cols: {df_all.shape[1]:,}")
print(f"- Base feature columns: {len(base_cols):,} | Lagged columns: {len(lag_cols):,}")
print(f"- Distinct regions (including Global if present): {len(regions):,}")
print(f"- Base features present: {features_found}")
print(f"- Remaining NaNs after fills: {nans_total}")
if skipped:
    print("Skipped regions:", ", ".join(skipped))


Saved weekly_ndbi_aligned.csv (NDBI features base + lags, fully filled).
Data validation summary:
- Date range: 2015-09-06 to 2025-09-28 | rows: 526 | cols: 100
- Base feature columns: 20 | Lagged columns: 80
- Distinct regions (including Global if present): 20
- Base features present: ['ndbi']
- Remaining NaNs after fills: 0
